# Track A (banking77) on Kaggle

Train and evaluate the banking77 intent classifier on a free Kaggle T4 (16 GB).
Everything is self-contained, you only upload the data.

## Setup (once)
1. Create a Kaggle Dataset from your local `track_a_banking77/data/` files:
   `labels.txt`, `gold.jsonl`, `seed.jsonl`, `train_synth.jsonl`. Name it
   **`track-a-data`** (it mounts at `/kaggle/input/track-a-data/`).
2. This notebook: **Settings -> Accelerator -> GPU T4**, and add the `track-a-data`
   dataset (right panel -> Add Input).
3. Run all cells top to bottom.

If your dataset slug differs, edit `DATA` in the config cell.


In [ ]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects; we use bitsandbytes
print("deps installed")

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
DATA = "/kaggle/input/track-a-data"   # edit if your dataset slug differs
OUT  = "/kaggle/working"
print("DATA contents:", os.listdir(DATA) if os.path.isdir(DATA) else "NOT FOUND - add the dataset")

In [ ]:
import json, re

def read_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]
def load_labels(path):
    with open(path) as f: return [l.strip() for l in f if l.strip()]
def system_prompt(labels):
    return ("You are a banking customer-support intent classifier. Read the customer "
            "message and reply with EXACTLY ONE intent label from the list below, copied "
            "verbatim, with nothing else (no quotes, no punctuation, no explanation).\n"
            "Allowed intents: " + ", ".join(labels))
def user_of(r): return next(m["content"] for m in r["messages"] if m["role"] == "user")
def assistant_of(r): return next(m["content"] for m in r["messages"] if m["role"] == "assistant")
def normalize(s): return " ".join(s.lower().split())
def predict_label(output, labels):
    by = {normalize(l): l for l in labels}
    first = normalize(output.splitlines()[0]) if output.strip() else ""
    if first in by: return by[first]
    low = normalize(output); hits = [l for l in labels if normalize(l) in low]
    return max(hits, key=len) if hits else None
def macro_f1(gold, pred, labels):
    tot = 0.0
    for l in labels:
        tp = sum(1 for g,p in zip(gold,pred) if g==l and p==l)
        fp = sum(1 for g,p in zip(gold,pred) if g!=l and p==l)
        fn = sum(1 for g,p in zip(gold,pred) if g==l and p!=l)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        tot += 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return tot/len(labels)

SENTINEL = [{"question": "What is the capital of France?", "answers": ["paris"]}, {"question": "What is 7 times 8?", "answers": ["56"]}, {"question": "Who wrote the play Romeo and Juliet?", "answers": ["shakespeare"]}, {"question": "What is the chemical symbol for water?", "answers": ["h2o"]}, {"question": "How many continents are there on Earth?", "answers": ["7", "seven"]}, {"question": "What planet is known as the Red Planet?", "answers": ["mars"]}, {"question": "What is the largest ocean on Earth?", "answers": ["pacific"]}, {"question": "In what year did World War II end?", "answers": ["1945"]}, {"question": "What gas do plants absorb from the air during photosynthesis?", "answers": ["carbon dioxide", "co2"]}, {"question": "What is the square root of 144?", "answers": ["12"]}, {"question": "What language is mainly spoken in Brazil?", "answers": ["portuguese"]}, {"question": "How do you say 'hello' in Spanish?", "answers": ["hola"]}]
print("helpers ready,", len(SENTINEL), "sentinel probes")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # same base the local evaluator scores

def load_model(adapter=None):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m = PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def gen(model, tok, prompts, max_new, batch=32):
    outs = []
    for i in range(0, len(prompts), batch):
        ch = prompts[i:i+batch]
        txt = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in ch]
        enc = tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)):
            outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
        print(f"  {min(i+batch,len(prompts))}/{len(prompts)}", end="\r")
    print()
    return outs
print("generation helpers ready")

## Train the two adapters (control = seed, real = seed-synth)

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(data_file, name, max_len=768, batch=4, grad_accum=4, epochs=3, lr=2e-4):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto")
    model.config.use_cache = False
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules="all-linear")
    ds = load_dataset("json", data_files=f"{DATA}/{data_file}", split="train")
    out = f"{OUT}/lora-{name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=batch,
        gradient_accumulation_steps=grad_accum, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy="epoch", bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=max_len, packing=False, assistant_only_loss=True, seed=0, report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
    r = tr.train(); tr.save_model(out); tok.save_pretrained(out)
    print(f"{name}: final_train_loss={r.training_loss:.4f} -> {out}")
    return out

train_one("seed.jsonl", "seed")              # control
train_one("train_synth.jsonl", "seed-synth") # real run

## Evaluate base vs seed vs seed-synth on both axes

In [ ]:
def evaluate(name, adapter=None):
    labels = load_labels(f"{DATA}/labels.txt")
    gold = read_jsonl(f"{DATA}/gold.jsonl")
    m, tok = load_model(adapter)
    print(f"[{name}] task on {len(gold)} gold rows")
    prompts = [[{"role":"system","content":system_prompt(labels)},
                {"role":"user","content":user_of(r)}] for r in gold]
    raw = gen(m, tok, prompts, 24)
    g = [assistant_of(r) for r in gold]; p = [predict_label(o, labels) for o in raw]
    n = len(gold); acc = sum(1 for a,b in zip(g,p) if a==b)/n
    valid = sum(1 for x in p if x is not None)/n; f1 = macro_f1(g, p, labels)
    sp = [[{"role":"user","content":x["question"]}] for x in SENTINEL]
    sraw = gen(m, tok, sp, 32)
    ok = sum(1 for x,o in zip(SENTINEL, sraw) if any(a in o.lower() for a in x["answers"]))
    res = {"name":name, "accuracy":round(acc,3), "macro_f1":round(f1,3),
           "valid_label_rate":round(valid,3), "sentinel":f"{ok}/{len(SENTINEL)}"}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2); print(" ", res)
    del m; import torch; torch.cuda.empty_cache()
    return res

R = [evaluate("base"),
     evaluate("seed", f"{OUT}/lora-seed"),
     evaluate("seed-synth", f"{OUT}/lora-seed-synth")]
print("\nname           acc    macroF1  valid  sentinel")
for r in R:
    print(f"{r['name']:<14} {r['accuracy']:.3f}  {r['macro_f1']:.3f}    {r['valid_label_rate']:.3f}  {r['sentinel']}")

## Download the adapters + results

In [ ]:
import shutil
for n in ["seed", "seed-synth"]:
    shutil.make_archive(f"{OUT}/lora-{n}", "zip", f"{OUT}/lora-{n}")
print("Download lora-seed.zip / lora-seed-synth.zip and result_*.json from the Output tab.")
print("Unzip each into phase3_train/lora-<name>/ locally, then re-run eval there if you want.")